In [ ]:
from utilities.settings import Settings
from utilities.models import Model
from utilities.tools import ToolCreation, Tool, Property, Parameter
from utilities.notifications import Notification
import gradio as gr
import json
from typing import Any
from datetime import datetime




In [ ]:
tool = ToolCreation()
notification = Notification()

In [ ]:
cvs = tool.read_pdf("./knowledge")

In [ ]:
vector_store = tool.create_embeddings(cvs);


In [ ]:
name = "Naheem Quadri"

In [ ]:
get_availability_tool = Tool(
    name="get_cal_availability",
    description="Check for available meeting slots on the calendar for a specific date range.",
    parameters=Parameter(
        properties=[
            Property(
                name="start_date",
                type="string",
                description="The start date to check in YYYY-MM-DD format.",
            ),
            Property(
                name="end_date",
                type="string",
                description="The end date to check in YYYY-MM-DD format.",
            ),
        ],
        required=["start_date", "end_date"],
    ),
)

send_email_tool = Tool(
    name="send_email",
    description=f"""Send an email notification to {name}. 
    For new visitor emails, only call this AFTER the user has explicitly provided 
    their name and email address in the conversation. Never call with empty or unknown values.""",
    parameters=Parameter(
        properties=[
            Property(
                name="subject",
                type="string",
                description="The subject line of the email.",
            ),
            Property(
                name="message",
                type="string",
                description="The body content of the email to send. Must include the user's name and email address.",
            ),
        ],
        required=["subject", "message"],
    ),
)



tool.create_tool(get_availability_tool, tool.get_cal_availability)
custom_tool = tool.create_tool(send_email_tool, notification.send_email)

print(f"Tools registered: {[t['function']['name'] for t in custom_tool]}")


In [ ]:


system_prompt = """
You are {name}, a Senior Backend Engineer and fintech specialist based in Lagos, Nigeria.
Today's date is {current_date}.
You are speaking directly as {name} using the email {owner_email} — never refer to yourself in third person.

## Your Role
Respond to professionals, potential employers, and collaborators inquiring about your career, 
skills, and accomplishments. Be confident, concise, and engaging at all times.

## CV Context
{contexts}

## Behavior Rules
- Speak in first person ("I built...", "I led...", "My experience...")
- Be confident and results-oriented — lead with impact and numbers where possible
- At the start of every new conversation, warmly ask for the user's name and email before proceeding
- DO NOT call send_email until the user has actually provided both their name and email
- Only call send_email("New Visitor...) AFTER the user has explicitly shared their name and email in the conversation
- Never admit uncertainty or limitations — handle unknown questions silently via `send_email`

## Tools Available

All tool calls are handled silently. User must not know about the tool calls and their details.

### `send_email(subject, message)`
Use this to keep {name} informed in real time. Call it for these four scenarios:

1. **New visitor** — when the user provides their name and email:
   - Subject: "New Visitor: <user name> just started a conversation"
   - Body: their name, email, and current date/time

2. **Unknown question** — when you cannot answer something, never expose the gap to the user:
   - Subject: "Unanswered Question from Visitor"
   - Body: the question and user's contact details if already collected

3. **User request forwarding** — when user says "tell Naheem...", "let him know...", "please inform him...":
   - Subject: "Message from Visitor: <user name>"
   - Body: their exact message and contact details if available

4. **Hiring or collaboration interest** — when a user hints at hiring or working together:
   - Subject: "Potential Opportunity from <user name>"
   - Body: summary of their interest and their contact details

### `get_cal_availability(start_date, end_date)`
- Call this whenever a user asks about scheduling, booking a call, or meeting availability
- Dates must be in "YYYY-MM-DD" format
- If no date range is specified, default to the next 7 days from today ({current_date})
- Present slots in a friendly format — e.g. "I'm free on Monday March 23rd at 9:00 AM, 10:00 AM..."
- Always include the booking link: https://cal.com/quadri-naheem-xbbrz5
"""

In [ ]:
model = Model(type="openrouter")

In [ ]:

def chat(message, history):
    contextual_data = "\n\n".join(tool.retrieve_context(message))
    
    messages = [{"role": "system", "content": system_prompt.format(
    name=name,
    current_date=datetime.now().strftime("%A, %B %d, %Y"),
    owner_email="naheemquadri3410@gmail.com",
    contexts=contextual_data,
)}]
    messages += history
    print(f"gradio history messages: {messages}")

    messages.append({"role": "user", "content": message})

    done = False
    response = None
    while not done:
        response = model.get_model(
            model_name="openai/gpt-4o-mini",
            messages=messages,
            tools=custom_tool
        )

        finish_reason = response.choices[0].finish_reason
        print(f"finish_reason: {finish_reason}") 
        if finish_reason == "tool_calls":
            assistant_message = response.choices[0].message
            tool_calls = getattr(assistant_message, "tool_calls", None)

            messages.append(assistant_message)

            tool_results = []

            if tool_calls:
                for tool_call in tool_calls:
                    try:
                        function = getattr(tool_call, "function", None)

                        if function is not None:
                            tool_name = function.name
                            raw_args = function.arguments
                        else:
                            tool_name = getattr(tool_call, "name", None)
                            raw_args = getattr(tool_call, "arguments", "{}")

                        try:
                            tool_args = json.loads(raw_args)
                        except Exception as e:
                            print("JSON ERROR:", e, raw_args)
                            continue

                        result = tool.handle_tool_call(tool_name, tool_args)

                        tool_results.append({
                            "role": "tool",
                            "tool_call_id": tool_call.id,
                            "content": result
                        })

                    except Exception as e:
                        print("TOOL ERROR:", e)
                        continue

                messages.extend(tool_results)
            else:
                done = True
        else:
            done = True

    if response is None:
        raise RuntimeError("No response generated from model.")
    
    return tool.evaluate_response(messages, response, name)

   

In [ ]:
gr.ChatInterface(chat, type="messages").launch()